# Attempt to forecast the price of MSFT by analyzing the prices of multiple stocks, including MSFT, over several consecutive days leading up to the target day.
#### N.B. Different setup from HW1

In [2]:
from torch.utils.data import DataLoader,Dataset

class StockDataset(Dataset):
    def __init__(self,X,Y,days):
        self.X = X
        self.Y = Y.reshape(-1)
        self.days = days # days ahead for prediction

    def __len__(self):
        return (len(self.Y)-self.days)

    def __getitem__(self,index):
        x=self.X[:,index:index+self.days]
        y=self.Y[index+self.days]
        return x,y



In [3]:
# !pip install pandas
# !pip install yfinance
import numpy as np
from numpy import exp, sum, log, log10
import yfinance as yf
import pandas as pd

def get_price(tick,start='2020-01-01',end=None):
    return yf.Ticker(tick).history(start=start,end=end)['Close']

def get_prices(tickers,start='2020-01-01',end=None):
    df=pd.DataFrame()
    for s in tickers:
        df[s]=get_price(s,start,end)
    return df

feature_stocks=['tsla','meta','nvda','amzn','nflx','gbtc','gdx','intc','dal','c','goog','aapl','msft','ibm','hp','orcl','sap','crm','hubs','twlo']
predict_stock='msft'

# getting data
start_date='2020-01-01'

allX=get_prices(feature_stocks,start=start_date)
ally=get_prices([predict_stock],start=start_date)

In [4]:
import torch.utils.data as data
import torch

stockData = StockDataset(allX.to_numpy().transpose().astype(np.float32),ally.to_numpy().astype(np.float32),days=5)
train_set_size = int(len(stockData)*0.7)
valid_set_size = int(len(stockData)*0.2)
test_set_size = len(stockData)-train_set_size-valid_set_size

train_set, valid_set, test_set = data.random_split(stockData,[train_set_size,valid_set_size,test_set_size],\
                                              generator=torch.Generator().manual_seed(42))

batch_size = train_set_size # use entire dataset as batch
train_dataloader = DataLoader(train_set,batch_size=batch_size,shuffle=True)  # input:(20,5), label:1
valid_dataloader = DataLoader(valid_set,batch_size=batch_size,shuffle=False)
test_dataloader = DataLoader(test_set,batch_size=batch_size,shuffle=False)

# 1. Build a simple MLP to forecast MSFT price using PyTorch Lightning.

#### You have total freedom of your MLP. But your MLP should take the last five day ($5 \times 20=100$) prices as input and you have to add dropout into your network.

## 1a. Create a subclass of pytorch_lightning.LightningModule. It should include \_\_init\_\_, training_step, validation_step, configure_optimizers in the class. (6 points)

## 1b. Create a subclass of pytorch_lightning.LightningDataModule. It should include \_\_init\_\_, train_dataloader, and val_dataloader in the class. (4 points)

## 1c. Complete the rest of the code and train the model with 70% of the data. You should set aside 15% of the data each for validation and testing.  Show the training and validation MSE (5 points)

# 2. Construct a 1-D CNN to forecast MSFT stock price. You are free to use any design, but your network must consist of at least one convolutional layer and one dropout layer. You can also extend the duration leading up to the target day by modifying the "days" argument in the StockDataset. But "days" should not be larger than 32. (10 points)


# 3. Please try to enhance the performance of the previously created MLP or CNN by applying hyperparameter tuning. You can use tools such as W&B hyperparameter sweep, SMAP, Optuna, or similar packages to achieve this. You need to optimize at least two parameters, with the dropout rate being one of them. (5 points)












In [11]:
import os
import torch

def save_best_model_as_pth(lightning_model, ckpt_path, pth_path):
    """
    Load the best Lightning checkpoint and save its state_dict as .pth
    """
    best_model = type(lightning_model).load_from_checkpoint(ckpt_path)
    torch.save(best_model.state_dict(), pth_path)
    print(f"Saved best checkpoint: {ckpt_path}")
    print(f"Saved .pth weights: {pth_path}")

In [12]:
# task1
import numpy as np
import pandas as pd
import yfinance as yf
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import pytorch_lightning as pl
import os
from pytorch_lightning.callbacks import ModelCheckpoint


torch.manual_seed(42)
np.random.seed(42)


def get_price(tick, start='2020-01-01', end=None):
    return yf.Ticker(tick).history(start=start, end=end)['Close']

def get_prices(tickers, start='2020-01-01', end=None):
    df = pd.DataFrame()
    for s in tickers:
        df[s] = get_price(s, start, end)
    return df

class StockDatasetMLP(Dataset):
    def __init__(self, X, Y, days=5):
        """
        X: shape [T, num_features]
        Y: shape [T]
        """
        self.X = X.astype(np.float32)
        self.Y = Y.astype(np.float32).reshape(-1)
        self.days = days

    def __len__(self):
        return len(self.Y) - self.days

    def __getitem__(self, index):
        # x_window shape: [days, 20]
        x_window = self.X[index:index + self.days]

        # flatten to 5 * 20 = 100
        x = torch.tensor(x_window.reshape(-1), dtype=torch.float32)

        # predict the next day
        y = torch.tensor(self.Y[index + self.days], dtype=torch.float32)

        return x, y


class StockMLPDataModule(pl.LightningDataModule):
    def __init__(self, feature_stocks, predict_stock, start_date='2020-01-01',
                 days=5, batch_size=32):
        super().__init__()
        self.feature_stocks = feature_stocks
        self.predict_stock = predict_stock
        self.start_date = start_date
        self.days = days
        self.batch_size = batch_size

    def setup(self, stage=None):
        allX = get_prices(self.feature_stocks, start=self.start_date)
        ally = get_prices([self.predict_stock], start=self.start_date)

        # align dates and remove missing rows
        data = allX.copy()
        data['target'] = ally[self.predict_stock]
        data = data.dropna().copy()

        X_all = data[self.feature_stocks].to_numpy(dtype=np.float32)   # [T, 20]
        Y_all = data['target'].to_numpy(dtype=np.float32)              # [T]

        # chronological split: 70 / 15 / 15
        T = len(data)
        train_end = int(T * 0.70)
        val_end = int(T * 0.85)

        X_train_raw = X_all[:train_end]
        X_val_raw = X_all[train_end:val_end]
        X_test_raw = X_all[val_end:]

        Y_train_raw = Y_all[:train_end]
        Y_val_raw = Y_all[train_end:val_end]
        Y_test_raw = Y_all[val_end:]

        # standardize using training statistics only
        x_mean = X_train_raw.mean(axis=0, keepdims=True)
        x_std = X_train_raw.std(axis=0, keepdims=True) + 1e-8

        y_mean = Y_train_raw.mean()
        y_std = Y_train_raw.std() + 1e-8

        X_train = (X_train_raw - x_mean) / x_std
        X_val = (X_val_raw - x_mean) / x_std
        X_test = (X_test_raw - x_mean) / x_std

        Y_train = (Y_train_raw - y_mean) / y_std
        Y_val = (Y_val_raw - y_mean) / y_std
        Y_test = (Y_test_raw - y_mean) / y_std

        self.y_mean = y_mean
        self.y_std = y_std

        self.train_dataset = StockDatasetMLP(X_train, Y_train, days=self.days)
        self.val_dataset = StockDatasetMLP(X_val, Y_val, days=self.days)
        self.test_dataset = StockDatasetMLP(X_test, Y_test, days=self.days)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)


class MLPForecaster(pl.LightningModule):
    def __init__(self, input_dim=100, hidden_dim1=128, hidden_dim2=64,
                 dropout=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim2, 1)
        )

        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.model(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        val_loss = self.criterion(y_hat, y)
        self.log("val_mse", val_loss, on_step=False, on_epoch=True, prog_bar=True)
        return val_loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        test_loss = self.criterion(y_hat, y)
        self.log("test_mse", test_loss, on_step=False, on_epoch=True, prog_bar=True)
        return test_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


feature_stocks = [
    'tsla', 'meta', 'nvda', 'amzn', 'nflx',
    'gbtc', 'gdx', 'intc', 'dal', 'c',
    'goog', 'aapl', 'msft', 'ibm', 'hpq',
    'orcl', 'sap', 'crm', 'hubs', 'twlo'
]
predict_stock = 'msft'

dm = StockMLPDataModule(
    feature_stocks=feature_stocks,
    predict_stock=predict_stock,
    start_date='2020-01-01',
    days=5,
    batch_size=32
)

model = MLPForecaster(
    input_dim=5 * 20,
    hidden_dim1=128,
    hidden_dim2=64,
    dropout=0.3,
    lr=1e-3
)


# save pth
os.makedirs("checkpoints/task1_mlp", exist_ok=True)

task1_checkpoint = ModelCheckpoint(
    dirpath="checkpoints/task1_mlp",
    filename="best_mlp",
    monitor="val_mse",
    mode="min",
    save_top_k=1
)

trainer = pl.Trainer(
    max_epochs=50,
    callbacks=[task1_checkpoint],
    logger=False
)

trainer.fit(model, datamodule=dm)

fit_metrics = trainer.callback_metrics.copy()
train_mse_std = fit_metrics["train_mse"].item() if "train_mse" in fit_metrics else None
val_mse_std = fit_metrics["val_mse"].item() if "val_mse" in fit_metrics else None

test_results = trainer.test(model, datamodule=dm)
test_mse_std = test_results[0]["test_mse"]

print("Standardized-scale Results")
print("Train MSE:", train_mse_std)
print("Val MSE:", val_mse_std)
print("Test MSE:", test_mse_std)

best_task1_ckpt = task1_checkpoint.best_model_path
print("Best Task 1 checkpoint:", best_task1_ckpt)

save_best_model_as_pth(
    lightning_model=model,
    ckpt_path=best_task1_ckpt,
    pth_path="checkpoints/task1_mlp/best_mlp.pth"
)

# compute original-scale test MSE
model.eval()
preds_std = []
targets_std = []

with torch.no_grad():
    for xb, yb in dm.test_dataloader():
        yhat = model(xb)
        preds_std.append(yhat.cpu().numpy())
        targets_std.append(yb.cpu().numpy())

preds_std = np.concatenate(preds_std)
targets_std = np.concatenate(targets_std)

preds_orig = preds_std * dm.y_std + dm.y_mean
targets_orig = targets_std * dm.y_std + dm.y_mean

test_mse_orig = np.mean((preds_orig - targets_orig) ** 2)

print("\nOriginal-price-scale Result")
print("Test MSE:", test_mse_orig)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/checkpoints/task1_mlp exists and is not empty.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ Sequential │ 21.2 K │ train │     0 │
│ 1 │ criterion │ MSELoss    │      0 │ train │     0 │
└───┴───────────┴────────────┴────────┴───────┴───────┘

Trainable params: 21.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mse          │    0.46007442474365234    │
└───────────────────────────┴───────────────────────────┘

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Standardized-scale Results
Train MSE: 0.11666932702064514
Val MSE: 0.4819351136684418
Test MSE: 0.46007442474365234
Best Task 1 checkpoint: /content/checkpoints/task1_mlp/best_mlp-v1.ckpt
Saved best checkpoint: /content/checkpoints/task1_mlp/best_mlp-v1.ckpt
Saved .pth weights: checkpoints/task1_mlp/best_mlp.pth

Original-price-scale Result
Test MSE: 2018.4475


In [9]:

# Task 2: 1D CNN for MSFT Price Forecasting


# !pip install yfinance pytorch-lightning

import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import pytorch_lightning as pl
import os
from pytorch_lightning.callbacks import ModelCheckpoint


torch.manual_seed(42)
np.random.seed(42)


def get_price(tick, start='2020-01-01', end=None):
    return yf.Ticker(tick).history(start=start, end=end)['Close']

def get_prices(tickers, start='2020-01-01', end=None):
    df = pd.DataFrame()
    for s in tickers:
        df[s] = get_price(s, start, end)
    return df

# 20 input stocks
feature_stocks = [
    'tsla', 'meta', 'nvda', 'amzn', 'nflx',
    'gbtc', 'gdx', 'intc', 'dal', 'c',
    'goog', 'aapl', 'msft', 'ibm', 'hpq',
    'orcl', 'sap', 'crm', 'hubs', 'twlo'
]

predict_stock = 'msft'
start_date = '2020-01-01'

# Download data
allX = get_prices(feature_stocks, start=start_date)
ally = get_prices([predict_stock], start=start_date)

# Align dates and remove missing rows
data = allX.copy()
data['target'] = ally[predict_stock]
data = data.dropna().copy()

print("Data shape after alignment and dropna:", data.shape)
print(data.head())


# X_all shape: [T, 20]
# Y_all shape: [T]
X_all = data[feature_stocks].to_numpy(dtype=np.float32)
Y_all = data['target'].to_numpy(dtype=np.float32)


# 4. Chronological split: 70 / 15 / 15

T = len(data)
train_end = int(T * 0.70)
val_end = int(T * 0.85)

X_train_raw = X_all[:train_end]
X_val_raw = X_all[train_end:val_end]
X_test_raw = X_all[val_end:]

Y_train_raw = Y_all[:train_end]
Y_val_raw = Y_all[train_end:val_end]
Y_test_raw = Y_all[val_end:]

print("\nSplit sizes:")
print("Train:", len(X_train_raw))
print("Val:", len(X_val_raw))
print("Test:", len(X_test_raw))



# Standardize X feature-wise
x_mean = X_train_raw.mean(axis=0, keepdims=True)           # shape [1, 20]
x_std = X_train_raw.std(axis=0, keepdims=True) + 1e-8

# Standardize Y
y_mean = Y_train_raw.mean()
y_std = Y_train_raw.std() + 1e-8

X_all_std = (X_all - x_mean) / x_std
Y_all_std = (Y_all - y_mean) / y_std


class StockDatasetCNN(Dataset):
    def __init__(self, X, Y, days=10):
        """
        X: shape [T, num_features]
        Y: shape [T]
        days: number of historical days used for prediction
        """
        self.X = X.astype(np.float32)
        self.Y = Y.astype(np.float32).reshape(-1)
        self.days = days

    def __len__(self):
        return len(self.Y) - self.days

    def __getitem__(self, index):
        # Window over the past "days" rows
        # x_window shape before transpose: [days, 20]
        x_window = self.X[index:index + self.days]

        # For Conv1d, input shape should be [channels, seq_len] = [20, days]
        x = torch.tensor(x_window.T, dtype=torch.float32)

        # Predict the next day after the window
        y = torch.tensor(self.Y[index + self.days], dtype=torch.float32)

        return x, y


class StockCNNDataModule(pl.LightningDataModule):
    def __init__(self, X_std, Y_std, days=10, batch_size=32):
        super().__init__()
        self.X_std = X_std
        self.Y_std = Y_std
        self.days = days
        self.batch_size = batch_size

    def setup(self, stage=None):
        full_dataset = StockDatasetCNN(self.X_std, self.Y_std, days=self.days)

        n = len(full_dataset)
        train_size = int(n * 0.70)
        val_size = int(n * 0.15)
        test_size = n - train_size - val_size

        self.train_dataset = Subset(full_dataset, range(0, train_size))
        self.val_dataset = Subset(full_dataset, range(train_size, train_size + val_size))
        self.test_dataset = Subset(full_dataset, range(train_size + val_size, n))

        print("\nDataset sizes after converting to sliding-window samples:")
        print("Train dataset:", len(self.train_dataset))
        print("Val dataset:", len(self.val_dataset))
        print("Test dataset:", len(self.test_dataset))

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)



class CNNForecaster(pl.LightningModule):
    def __init__(self, lr=1e-3, dropout=0.3):
        super().__init__()
        self.save_hyperparameters()

        # Input shape: [batch, 20, days]
        self.conv1 = nn.Conv1d(in_channels=20, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(64, 1)

        self.criterion = nn.MSELoss()

    def forward(self, x):
        # x: [batch, 20, days]
        x = self.conv1(x)          # [batch, 32, days]
        x = self.relu(x)
        x = self.dropout(x)
        x = self.pool(x)           # [batch, 32, 1]
        x = x.squeeze(-1)          # [batch, 32]
        x = self.fc1(x)            # [batch, 64]
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x).squeeze(-1)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("val_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("test_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

# hyperparameters
days = 10          # Must be <= 32
batch_size = 32
learning_rate = 1e-3
dropout_rate = 0.2
max_epochs = 80


dm = StockCNNDataModule(
    X_std=X_all_std,
    Y_std=Y_all_std,
    days=days,
    batch_size=batch_size
)

model = CNNForecaster(
    lr=learning_rate,
    dropout=dropout_rate
)




os.makedirs("checkpoints/task2_cnn", exist_ok=True)

task2_checkpoint = ModelCheckpoint(
    dirpath="checkpoints/task2_cnn",
    filename="best_cnn",
    monitor="val_mse",
    mode="min",
    save_top_k=1
)

trainer = pl.Trainer(
    max_epochs=80,
    callbacks=[task2_checkpoint],
    logger=False
)

trainer.fit(model, datamodule=dm)

# Save fit-stage metrics before test()
fit_metrics = trainer.callback_metrics.copy()
train_mse_std = fit_metrics["train_mse"].item() if "train_mse" in fit_metrics else None
val_mse_std = fit_metrics["val_mse"].item() if "val_mse" in fit_metrics else None


test_results = trainer.test(model, datamodule=dm)
test_mse_std = test_results[0]["test_mse"]

print("\n==============================")
print("Standardized-scale Results")
print("==============================")
print("Train MSE:", train_mse_std)
print("Val MSE:", val_mse_std)
print("Test MSE:", test_mse_std)

best_task2_ckpt = task2_checkpoint.best_model_path
print("Best Task 2 checkpoint:", best_task2_ckpt)

save_best_model_as_pth(
    CNNForecaster,
    best_task2_ckpt,
    "checkpoints/task2_cnn/best_cnn.pth"
)


model.eval()
preds_std = []
targets_std = []

test_loader = dm.test_dataloader()

with torch.no_grad():
    for xb, yb in test_loader:
        yhat = model(xb)
        preds_std.append(yhat.cpu().numpy())
        targets_std.append(yb.cpu().numpy())

preds_std = np.concatenate(preds_std)
targets_std = np.concatenate(targets_std)

# Inverse transform to original scale
preds_orig = preds_std * y_std + y_mean
targets_orig = targets_std * y_std + y_mean

test_mse_orig = np.mean((preds_orig - targets_orig) ** 2)

print("\n==============================")
print("Original-price-scale Result")
print("==============================")
print("Test MSE:", test_mse_orig)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Data shape after alignment and dropna: (1567, 21)
                                tsla        meta      nvda       amzn  \
Date                                                                    
2020-01-02 00:00:00-05:00  28.684000  208.146576  5.970755  94.900497   
2020-01-03 00:00:00-05:00  29.534000  207.045227  5.875187  93.748497   
2020-01-06 00:00:00-05:00  30.102667  210.944626  5.899826  95.143997   
2020-01-07 00:00:00-05:00  31.270666  211.401016  5.971252  95.343002   
2020-01-08 00:00:00-05:00  32.809334  213.544205  5.982451  94.598503   

                                nflx      gbtc        gdx       intc  \
Date                                                                   
2020-01-02 00:00:00-05:00  32.980999  7.208672  27.242304  53.666466   
2020-01-03 00:00:00-05:00  32.590000  7.759711  27.075232  53.013721   
2020-01-06 00:00:00-05:00  33.583000  8.121048  27.121639  52.863770   
2020-01-07 00:00:00-05:00  33.075001  9.123758  27.381531  51.981678   
2020-0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/checkpoints/task2_cnn exists and is not empty.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv1     │ Conv1d            │  2.0 K │ train │     0 │
│ 1 │ relu      │ ReLU              │      0 │ train │     0 │
│ 2 │ dropout   │ Dropout           │      0 │ train │     0 │
│ 3 │ pool      │ AdaptiveAvgPool1d │      0 │ train │     0 │
│ 4 │ fc1       │ Linear            │  2.1 K │ train │     0 │
│ 5 │ fc2       │ Linear            │     65 │ train │     0 │
│ 6 │ criterion │ MSELoss           │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 4.1 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.1 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=80` reached.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mse          │    0.6339601278305054     │
└───────────────────────────┴───────────────────────────┘


Dataset sizes after converting to sliding-window samples:
Train dataset: 1089
Val dataset: 233
Test dataset: 235



Standardized-scale Results
Train MSE: 0.09582676738500595
Val MSE: 1.0549732446670532
Test MSE: 0.6339601278305054
Best Task 2 checkpoint: /content/checkpoints/task2_cnn/best_cnn-v4.ckpt
Saved best checkpoint: /content/checkpoints/task2_cnn/best_cnn-v4.ckpt
Saved .pth weights: checkpoints/task2_cnn/best_cnn.pth

Original-price-scale Result
Test MSE: 2781.3228


In [4]:

# Task 3: Hyperparameter Tuning for CNN


!pip install yfinance pytorch-lightning optuna

import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import pytorch_lightning as pl
import optuna
import os
from pytorch_lightning.callbacks import ModelCheckpoint


torch.manual_seed(42)
np.random.seed(42)


def get_price(tick, start='2020-01-01', end=None):
    return yf.Ticker(tick).history(start=start, end=end)['Close']

def get_prices(tickers, start='2020-01-01', end=None):
    df = pd.DataFrame()
    for s in tickers:
        df[s] = get_price(s, start, end)
    return df

feature_stocks = [
    'tsla', 'meta', 'nvda', 'amzn', 'nflx',
    'gbtc', 'gdx', 'intc', 'dal', 'c',
    'goog', 'aapl', 'msft', 'ibm', 'hpq',
    'orcl', 'sap', 'crm', 'hubs', 'twlo'
]
predict_stock = 'msft'
start_date = '2020-01-01'

allX = get_prices(feature_stocks, start=start_date)
ally = get_prices([predict_stock], start=start_date)

# Align dates and remove missing rows
data = allX.copy()
data['target'] = ally[predict_stock]
data = data.dropna().copy()

print("Data shape after alignment and dropna:", data.shape)


X_all = data[feature_stocks].to_numpy(dtype=np.float32)   # [T, 20]
Y_all = data['target'].to_numpy(dtype=np.float32)         # [T]


T = len(data)
train_end = int(T * 0.70)
val_end = int(T * 0.85)

X_train_raw = X_all[:train_end]
X_val_raw = X_all[train_end:val_end]
X_test_raw = X_all[val_end:]

Y_train_raw = Y_all[:train_end]
Y_val_raw = Y_all[train_end:val_end]
Y_test_raw = Y_all[val_end:]

print("\nRaw split sizes:")
print("Train:", len(X_train_raw))
print("Val:", len(X_val_raw))
print("Test:", len(X_test_raw))


x_mean = X_train_raw.mean(axis=0, keepdims=True)
x_std = X_train_raw.std(axis=0, keepdims=True) + 1e-8

y_mean = Y_train_raw.mean()
y_std = Y_train_raw.std() + 1e-8

X_train_std = (X_train_raw - x_mean) / x_std
X_val_std = (X_val_raw - x_mean) / x_std
X_test_std = (X_test_raw - x_mean) / x_std

Y_train_std = (Y_train_raw - y_mean) / y_std
Y_val_std = (Y_val_raw - y_mean) / y_std
Y_test_std = (Y_test_raw - y_mean) / y_std


class StockDatasetCNN(Dataset):
    def __init__(self, X, Y, days=10):
        """
        X: [T, 20]
        Y: [T]
        """
        self.X = X.astype(np.float32)
        self.Y = Y.astype(np.float32).reshape(-1)
        self.days = days

    def __len__(self):
        return len(self.Y) - self.days

    def __getitem__(self, index):
        # x_window: [days, 20]
        x_window = self.X[index:index + self.days]

        # Conv1d input: [channels=20, seq_len=days]
        x = torch.tensor(x_window.T, dtype=torch.float32)
        y = torch.tensor(self.Y[index + self.days], dtype=torch.float32)
        return x, y


class StockCNNDataModule(pl.LightningDataModule):
    def __init__(self, X_train, Y_train, X_val, Y_val, X_test, Y_test,
                 days=10, batch_size=32):
        super().__init__()
        self.X_train = X_train
        self.Y_train = Y_train
        self.X_val = X_val
        self.Y_val = Y_val
        self.X_test = X_test
        self.Y_test = Y_test
        self.days = days
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_dataset = StockDatasetCNN(self.X_train, self.Y_train, days=self.days)
        self.val_dataset = StockDatasetCNN(self.X_val, self.Y_val, days=self.days)
        self.test_dataset = StockDatasetCNN(self.X_test, self.Y_test, days=self.days)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=False)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)


class TunableCNNForecaster(pl.LightningModule):
    def __init__(self, conv_channels=32, hidden_dim=64, lr=1e-3, dropout=0.3):
        super().__init__()
        self.save_hyperparameters()

        self.conv1 = nn.Conv1d(
            in_channels=20,
            out_channels=conv_channels,
            kernel_size=3,
            padding=1
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(conv_channels, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

        self.criterion = nn.MSELoss()

    def forward(self, x):
        # x: [batch, 20, days]
        x = self.conv1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.pool(x)
        x = x.squeeze(-1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x).squeeze(-1)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_mse", loss, on_step=False, on_epoch=True, prog_bar=False)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("val_mse", loss, on_step=False, on_epoch=True, prog_bar=False)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("test_mse", loss, on_step=False, on_epoch=True, prog_bar=False)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


def objective(trial):
    days = trial.suggest_int("days", 5, 32)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    conv_channels = trial.suggest_categorical("conv_channels", [16, 32, 64, 128])
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128, 256])
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    dm = StockCNNDataModule(
        X_train=X_train_std,
        Y_train=Y_train_std,
        X_val=X_val_std,
        Y_val=Y_val_std,
        X_test=X_test_std,
        Y_test=Y_test_std,
        days=days,
        batch_size=batch_size
    )

    model = TunableCNNForecaster(
        conv_channels=conv_channels,
        hidden_dim=hidden_dim,
        lr=lr,
        dropout=dropout
    )

    trainer = pl.Trainer(
        max_epochs=50,
        enable_checkpointing=False,
        logger=False,
        enable_progress_bar=False
    )

    trainer.fit(model, datamodule=dm)

    metrics = trainer.callback_metrics
    if "val_mse" not in metrics:
        raise ValueError(f"val_mse not found in callback_metrics. Available keys: {list(metrics.keys())}")

    val_mse = metrics["val_mse"].item()
    return val_mse

from pytorch_lightning.callbacks import ModelCheckpoint
import os
import torch

def save_best_model_as_pth(lightning_model_class, ckpt_path, pth_path):
    best_model = lightning_model_class.load_from_checkpoint(ckpt_path)
    torch.save(best_model.state_dict(), pth_path)
    print("Saved best checkpoint:", ckpt_path)
    print("Saved .pth weights:", pth_path)

# Run Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("\n===================================")
print("Best trial from Optuna")
print("===================================")
print("Best validation MSE:", study.best_value)
print("Best parameters:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

# Retrain best model
best_params = study.best_params

best_dm = StockCNNDataModule(
    X_train=X_train_std,
    Y_train=Y_train_std,
    X_val=X_val_std,
    Y_val=Y_val_std,
    X_test=X_test_std,
    Y_test=Y_test_std,
    days=best_params["days"],
    batch_size=best_params["batch_size"]
)

best_model = TunableCNNForecaster(
    conv_channels=best_params["conv_channels"],
    hidden_dim=best_params["hidden_dim"],
    lr=best_params["lr"],
    dropout=best_params["dropout"]
)

os.makedirs("checkpoints/task3_tuned_cnn", exist_ok=True)

task3_checkpoint = ModelCheckpoint(
    dirpath="checkpoints/task3_tuned_cnn",
    filename="best_tuned_cnn",
    monitor="val_mse",
    mode="min",
    save_top_k=1
)

best_trainer = pl.Trainer(
    max_epochs=80,
    callbacks=[task3_checkpoint],
    logger=False
)

best_trainer.fit(best_model, datamodule=best_dm)

fit_metrics = best_trainer.callback_metrics.copy()
best_train_mse_std = fit_metrics["train_mse"].item() if "train_mse" in fit_metrics else None
best_val_mse_std = fit_metrics["val_mse"].item() if "val_mse" in fit_metrics else None

test_results = best_trainer.test(best_model, datamodule=best_dm)
best_test_mse_std = test_results[0]["test_mse"]

best_task3_ckpt = task3_checkpoint.best_model_path
print("Best Task 3 checkpoint:", best_task3_ckpt)

save_best_model_as_pth(
    TunableCNNForecaster,
    best_task3_ckpt,
    "checkpoints/task3_tuned_cnn/best_tuned_cnn.pth"
)

print("\n===================================")
print("Best tuned CNN results")
print("===================================")
print("Train MSE (standardized):", best_train_mse_std)
print("Val MSE (standardized):", best_val_mse_std)
print("Test MSE (standardized):", best_test_mse_std)

[I 2026-03-30 03:43:37,025] A new study created in memory with name: no-name-cb78574e-60b1-461b-a0a7-346885a20aa0
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Data shape after alignment and dropna: (1567, 21)

Raw split sizes:
Train: 1096
Val: 235
Test: 236


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv1     │ Conv1d            │  3.9 K │ train │     0 │
│ 1 │ relu      │ ReLU              │      0 │ train │     0 │
│ 2 │ dropout   │ Dropout           │      0 │ train │     0 │
│ 3 │ pool      │ AdaptiveAvgPool1d │      0 │ train │     0 │
│ 4 │ fc1       │ Linear            │  4.2 K │ train │     0 │
│ 5 │ fc2       │ Linear            │     65 │ train │     0 │
│ 6 │ criterion │ MSELoss           │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 8.1 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.1 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 7                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
[I 2026-03-30 03:43:56,492] Trial 0 finished with value: 0.15483225882053375 and parameters: {'days': 32, 'dropout': 0.3130129789980296, 'lr': 0.00035445527224186517, 'conv_channels': 64, 'hidden_dim': 64, 'batch_size': 16}. Best is trial 0 with value: 0.15483225882053375.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=80` reached.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_mse          │    0.14584693312644958    │
└───────────────────────────┴───────────────────────────┘

Best Task 3 checkpoint: /content/checkpoints/task3_tuned_cnn/best_tuned_cnn-v3.ckpt
Saved best checkpoint: /content/checkpoints/task3_tuned_cnn/best_tuned_cnn-v3.ckpt
Saved .pth weights: checkpoints/task3_tuned_cnn/best_tuned_cnn.pth

Best tuned CNN results
Train MSE (standardized): 0.042297929525375366
Val MSE (standardized): 0.45049503445625305
Test MSE (standardized): 0.14584693312644958
